In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ============================================================
#  OriginScale Fair Benchmark  —  single Kaggle cell (fixed)
# ============================================================

!pip install -q hdbscan

import numpy as np, pandas as pd, time, json, warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans, MiniBatchKMeans, Birch, AgglomerativeClustering, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs, make_moons, make_circles, load_iris, load_wine, load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import hdbscan as HDBSCAN

# ── KMedoids (pure numpy, no sklearn_extra needed) ───────────
class KMedoids:
    def __init__(self, n_clusters=3, max_iter=300, random_state=42):
        self.n_clusters, self.max_iter = n_clusters, max_iter
        np.random.seed(random_state)

    def fit(self, X):
        idx     = np.random.choice(len(X), self.n_clusters, replace=False)
        medoids = idx.copy()
        for _ in range(self.max_iter):
            dists  = np.linalg.norm(X[:, None] - X[medoids][None], axis=2)
            labels = np.argmin(dists, axis=1)
            new_medoids = medoids.copy()
            for c in range(self.n_clusters):
                members = np.where(labels == c)[0]
                if len(members) == 0: continue
                sub = np.sum(np.linalg.norm(X[members][:, None] - X[members][None], axis=2), axis=1)
                new_medoids[c] = members[np.argmin(sub)]
            if np.all(new_medoids == medoids): break
            medoids = new_medoids
        self.labels_ = labels
        return self

# ── OriginScale ───────────────────────────────────────────────
class OriginScale:
    def __init__(self, n_clusters=3, tol=1e-4, max_iter=300):
        self.n_clusters, self.tol, self.max_iter = n_clusters, tol, max_iter

    def fit(self, X):
        idx       = np.argsort(np.linalg.norm(X, axis=1))
        centroids = X[idx[:self.n_clusters]].copy()
        for _ in range(self.max_iter):
            dists  = np.linalg.norm(X[:, None] - centroids[None], axis=2)
            labels = np.argmin(dists, axis=1)
            new_c  = np.array([X[labels==i].mean(axis=0) if (labels==i).any()
                               else X[np.random.randint(len(X))] for i in range(self.n_clusters)])
            if np.linalg.norm(new_c - centroids) < self.tol: break
            centroids = new_c
        self.labels_, self.centroids_ = labels, centroids
        return self

# ── Datasets ──────────────────────────────────────────────────
def get_datasets():
    ds = {}
    X, y = make_moons(1000, noise=0.1, random_state=42)
    ds['moons']         = (StandardScaler().fit_transform(X), y, 2)
    X, y = make_blobs(1000, centers=4, random_state=42)
    ds['blobs']         = (StandardScaler().fit_transform(X), y, 4)
    X, y = make_circles(1000, noise=0.1, factor=0.3, random_state=42)
    ds['circles']       = (StandardScaler().fit_transform(X), y, 2)
    X, y = make_blobs(1000, centers=3, random_state=42)
    X    = np.dot(X, [[0.8,0.2],[0.2,0.8]])
    ds['anisotropic']   = (StandardScaler().fit_transform(X), y, 3)
    ir = load_iris()
    ds['iris']          = (StandardScaler().fit_transform(ir.data),  ir.target,  3)
    wi = load_wine()
    ds['wine']          = (StandardScaler().fit_transform(wi.data),  wi.target,  3)
    ca = load_breast_cancer()
    ds['breast_cancer'] = (StandardScaler().fit_transform(ca.data),  ca.target,  2)
    return ds

# ── Algorithm factory (ALL centroid methods n_init=1) ─────────
def get_algorithms(k):
    return {
        'OriginScale':      lambda X: OriginScale(k).fit(X).labels_,
        'KMeans_random_1':  lambda X: KMeans(k, init='random',    n_init=1,  random_state=42).fit(X).labels_,
        'KMeans_pp_1':      lambda X: KMeans(k, init='k-means++', n_init=1,  random_state=42).fit(X).labels_,
        'KMeans_pp_10':     lambda X: KMeans(k, init='k-means++', n_init=10, random_state=42).fit(X).labels_,
        'MiniBatch_KMeans': lambda X: MiniBatchKMeans(k, n_init=1, random_state=42).fit(X).labels_,
        'Agglomerative':    lambda X: AgglomerativeClustering(k).fit_predict(X),
        'GMM':              lambda X: GaussianMixture(k, random_state=42).fit(X).predict(X),
        'BIRCH':            lambda X: Birch(n_clusters=k).fit(X).labels_,
        'Spectral':         lambda X: SpectralClustering(k, random_state=42, n_neighbors=10).fit_predict(X),
        'KMedoids':         lambda X: KMedoids(k, random_state=42).fit(X).labels_,
        'HDBSCAN':          lambda X: HDBSCAN.HDBSCAN(min_cluster_size=max(5, len(X)//100)).fit_predict(X),
    }

def score(X, labels):
    u = np.unique(labels[labels != -1])
    if len(u) < 2: return None, None, None
    Xf, lf = X[labels!=-1], labels[labels!=-1]
    try:    sil = round(float(silhouette_score(Xf, lf)), 4)
    except: sil = None
    try:    ch  = round(float(calinski_harabasz_score(Xf, lf)), 2)
    except: ch  = None
    try:    db  = round(float(davies_bouldin_score(Xf, lf)), 4)
    except: db  = None
    return sil, ch, db

REPS = 5

# ═══════════════════════════════════════════════════════════════
#  EXP 1 — Quality + fair timing on benchmark datasets
# ═══════════════════════════════════════════════════════════════
print("=" * 62)
print("EXP 1: Benchmark datasets  (n_init=1 for all centroid methods)")
print("=" * 62)

datasets  = get_datasets()
exp1_rows = []

for ds_name, (X, y, k) in datasets.items():
    algs = get_algorithms(k)
    for alg_name, alg_fn in algs.items():
        times, labels = [], None
        for _ in range(REPS):
            t0 = time.perf_counter()
            try:    labels = alg_fn(X)
            except: labels = None
            times.append(time.perf_counter() - t0)
        if labels is None: continue
        sil, ch, db = score(X, labels)
        row = dict(dataset=ds_name, algorithm=alg_name, k=k,
                   n_samples=len(X), n_features=X.shape[1],
                   silhouette=sil, calinski_harabasz=ch, davies_bouldin=db,
                   time_mean=round(np.mean(times), 6),
                   time_std=round(np.std(times),   6),
                   time_min=round(np.min(times),   6))
        exp1_rows.append(row)
        print(f"  {ds_name:15s} | {alg_name:20s} | sil={str(sil):6s} | t={np.mean(times):.5f}s")

# ═══════════════════════════════════════════════════════════════
#  EXP 2 — Off-center ablation
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 62)
print("EXP 2: Off-center ablation  (does origin-distance degrade?)")
print("=" * 62)

offsets           = [0, 2, 5, 10, 25, 50, 100]
exp2_rows         = []
ablation_ds       = {n: datasets[n] for n in ['blobs', 'moons', 'anisotropic', 'iris']}
ablation_alg_keys = ['OriginScale', 'KMeans_pp_1', 'KMeans_random_1']

for ds_name, (X, y, k) in ablation_ds.items():
    for offset in offsets:
        Xs   = X + offset
        algs = get_algorithms(k)
        for alg_name in ablation_alg_keys:
            try:
                labels = algs[alg_name](Xs)
                sil, _, _ = score(Xs, labels)
            except:
                sil = None
            row = dict(dataset=ds_name, offset=offset, algorithm=alg_name, silhouette=sil)
            exp2_rows.append(row)
            print(f"  {ds_name:15s} | offset={offset:5.0f} | {alg_name:20s} | sil={str(sil):6s}")

# ═══════════════════════════════════════════════════════════════
#  EXP 3 — Scalability (init time, n up to 500k)
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 62)
print("EXP 3: Scalability  (initialization time only)")
print("=" * 62)

exp3_rows  = []
SCALE_REPS = 3

def timed(fn, reps):
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return round(min(times), 6)

for n in [1_000, 5_000, 10_000, 50_000, 100_000, 500_000]:
    for k in [3, 10, 50, 100]:
        X_s, _ = make_blobs(n_samples=n, centers=k, random_state=42)
        X_s    = StandardScaler().fit_transform(X_s)

        t_os   = timed(lambda: X_s[np.argsort(np.linalg.norm(X_s, axis=1))[:k]].copy(), SCALE_REPS)
        t_pp   = timed(lambda: KMeans(k, init='k-means++', n_init=1, max_iter=1, random_state=42).fit(X_s), SCALE_REPS)
        t_rand = timed(lambda: KMeans(k, init='random',    n_init=1, max_iter=1, random_state=42).fit(X_s), SCALE_REPS)

        row = dict(n_samples=n, k=k,
                   originscale_s=t_os, kmeanspp_s=t_pp, random_s=t_rand,
                   speedup_vs_pp=  round(t_pp   / t_os, 2) if t_os > 0 else None,
                   speedup_vs_rand=round(t_rand / t_os, 2) if t_os > 0 else None)
        exp3_rows.append(row)
        print(f"  n={n:>7,}  k={k:>3} | OS={t_os:.5f}s  KM++={t_pp:.5f}s  speedup={row['speedup_vs_pp']}x")

# ═══════════════════════════════════════════════════════════════
#  EXP 4 — Reproducibility (20 unseeded runs)
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 62)
print("EXP 4: Reproducibility  (20 unseeded runs per algorithm)")
print("=" * 62)

REPRO_RUNS = 20
exp4_rows  = []
repro_ds   = {n: datasets[n] for n in ['blobs', 'iris', 'wine']}
repro_algs = {
    'OriginScale':     lambda X, k: OriginScale(k).fit(X).labels_,
    'KMeans_pp_1':     lambda X, k: KMeans(k, init='k-means++', n_init=1).fit(X).labels_,
    'KMeans_random_1': lambda X, k: KMeans(k, init='random',    n_init=1).fit(X).labels_,
}

for ds_name, (X, y, k) in repro_ds.items():
    for alg_name, alg_fn in repro_algs.items():
        scores = []
        for _ in range(REPRO_RUNS):
            try:
                sil, _, _ = score(X, alg_fn(X, k))
                if sil is not None: scores.append(sil)
            except: pass
        row = dict(dataset=ds_name, algorithm=alg_name,
                   mean=round(np.mean(scores), 4),
                   std= round(np.std(scores),  6),
                   min= round(np.min(scores),  4),
                   max= round(np.max(scores),  4),
                   variance=round(np.var(scores), 8))
        exp4_rows.append(row)
        print(f"  {ds_name:12s} | {alg_name:20s} | mean={np.mean(scores):.4f}  std={np.std(scores):.6f}")

# ═══════════════════════════════════════════════════════════════
#  SAVE + PRINT — paste the JSON output back to Claude
# ═══════════════════════════════════════════════════════════════
results = {
    "exp1_benchmark":       exp1_rows,
    "exp2_ablation":        exp2_rows,
    "exp3_scalability":     exp3_rows,
    "exp4_reproducibility": exp4_rows,
}

pd.DataFrame(exp1_rows).to_csv('exp1_benchmark.csv',       index=False)
pd.DataFrame(exp2_rows).to_csv('exp2_ablation.csv',        index=False)
pd.DataFrame(exp3_rows).to_csv('exp3_scalability.csv',     index=False)
pd.DataFrame(exp4_rows).to_csv('exp4_reproducibility.csv', index=False)

print("\n\n" + "=" * 62)
print("PASTE THIS JSON BACK TO CLAUDE TO GENERATE PAPER TABLES:")
print("=" * 62)
print(json.dumps(results, indent=2))

EXP 1: Benchmark datasets  (n_init=1 for all centroid methods)
  moons           | OriginScale          | sil=0.4904 | t=0.00273s
  moons           | KMeans_random_1      | sil=0.4904 | t=0.02119s
  moons           | KMeans_pp_1          | sil=0.4905 | t=0.00617s
  moons           | KMeans_pp_10         | sil=0.4904 | t=0.02364s
  moons           | MiniBatch_KMeans     | sil=0.4903 | t=0.01863s
  moons           | Agglomerative        | sil=0.4458 | t=0.02993s
  moons           | GMM                  | sil=0.4907 | t=0.02260s
  moons           | BIRCH                | sil=0.4515 | t=0.02766s
  moons           | Spectral             | sil=0.4901 | t=0.20309s
  moons           | KMedoids             | sil=0.4896 | t=0.12478s
  moons           | HDBSCAN              | sil=0.3851 | t=0.03383s
  blobs           | OriginScale          | sil=0.7983 | t=0.00352s
  blobs           | KMeans_random_1      | sil=0.7983 | t=0.00530s
  blobs           | KMeans_pp_1          | sil=0.7983 | t=0.00606s

In [3]:
# ============================================================
#  OriginScale — Large-Scale Experiment + Plots
#  Reviewer requirements:
#    - n >= 100k, k >= 50
#    - Init time vs k plot
#    - Final inertia comparison
#    - Convergence iterations plot
#    - Extra baselines: MiniBatch KMeans, PCA-based init (scalable)
#    - Complexity / memory analysis table
# ============================================================

!pip install -q hdbscan matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import time, json, warnings, tracemalloc, os
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid", palette="deep")
FIGDIR = "/kaggle/working"

# ══════════════════════════════════════════════════════════════
#  INITIALIZERS
# ══════════════════════════════════════════════════════════════

class OriginScale:
    """Origin-distance sort initializer + Lloyd iterations."""
    def __init__(self, n_clusters=3, tol=1e-4, max_iter=300):
        self.n_clusters, self.tol, self.max_iter = n_clusters, tol, max_iter

    def _init_centroids(self, X):
        idx = np.argsort(np.linalg.norm(X, axis=1))
        return X[idx[:self.n_clusters]].copy()

    def fit(self, X):
        centroids = self._init_centroids(X)
        inertia_history = []
        n_iter = 0
        for i in range(self.max_iter):
            n_iter = i + 1
            dists  = np.linalg.norm(X[:, None] - centroids[None], axis=2)
            labels = np.argmin(dists, axis=1)
            inertia = float(np.sum(np.min(dists, axis=1)**2))
            inertia_history.append(inertia)
            new_c = np.array([
                X[labels==j].mean(axis=0) if (labels==j).any()
                else X[np.random.randint(len(X))]
                for j in range(self.n_clusters)
            ])
            if np.linalg.norm(new_c - centroids) < self.tol:
                break
            centroids = new_c
        self.labels_         = labels
        self.centroids_      = centroids
        self.inertia_        = inertia_history[-1]
        self.inertia_history = inertia_history
        self.n_iter_         = n_iter
        return self


def pca_init(X, k):
    """
    PCA-based scalable initializer (recent baseline):
    Project onto first ceil(log2(k)) PCs, divide each axis at median,
    take mean of each partition as centroid. Fills remaining slots
    with furthest-first from those seeds. O(n * min(n,d) * n_components).
    Reference: Celebi et al. (2013) — a standard scalable deterministic init.
    """
    n_components = max(1, int(np.ceil(np.log2(k))))
    n_components = min(n_components, X.shape[1], X.shape[0])
    pca = PCA(n_components=n_components, random_state=42)
    Z   = pca.fit_transform(X)

    # Recursive binary split on PC axes to get ~k initial regions
    def split(indices, depth, budget):
        if len(indices) == 0 or budget <= 0:
            return []
        if budget == 1 or depth >= n_components:
            return [indices[np.random.randint(len(indices))]]
        ax     = depth % n_components
        median = np.median(Z[indices, ax])
        left   = indices[Z[indices, ax] <= median]
        right  = indices[Z[indices, ax] >  median]
        half   = budget // 2
        return split(left, depth+1, half) + split(right, depth+1, budget - half)

    all_idx   = np.arange(len(X))
    seed_idx  = split(all_idx, 0, k)
    # Pad with furthest-first if needed
    seed_idx  = list(seed_idx)
    used      = set(seed_idx)
    centroids = X[seed_idx].copy() if seed_idx else X[[0]].copy()
    while len(seed_idx) < k:
        d2   = np.min(np.linalg.norm(X[:, None] - centroids[None], axis=2), axis=1)
        best = int(np.argmax(np.where(
            [i in used for i in range(len(X))], -1, d2
        )))
        seed_idx.append(best)
        used.add(best)
        centroids = np.vstack([centroids, X[best]])
    return centroids[:k]


def run_full(X, k, init_fn, name, max_iter=300, tol=1e-4, reps=3):
    """
    Run full K-means from a given init function.
    Returns dict with timing, inertia history, iterations, memory.
    """
    best = None
    for _ in range(reps):
        # ── memory ──
        tracemalloc.start()
        t0         = time.perf_counter()
        centroids  = init_fn(X, k)
        t_init     = time.perf_counter() - t0
        _, mem_init = tracemalloc.get_traced_memory()
        tracemalloc.stop()

        # ── Lloyd iterations ──
        tracemalloc.start()
        t1            = time.perf_counter()
        inertia_hist  = []
        prev          = centroids.copy()
        labels        = None
        for i in range(max_iter):
            dists  = np.linalg.norm(X[:, None] - prev[None], axis=2)
            labels = np.argmin(dists, axis=1)
            inertia = float(np.sum(np.min(dists, axis=1)**2))
            inertia_hist.append(inertia)
            new_c = np.array([
                X[labels==j].mean(axis=0) if (labels==j).any()
                else X[np.random.randint(len(X))]
                for j in range(k)
            ])
            if np.linalg.norm(new_c - prev) < tol:
                break
            prev = new_c
        t_lloyd = time.perf_counter() - t1
        _, mem_lloyd = tracemalloc.get_traced_memory()
        tracemalloc.stop()

        result = dict(
            name=name, k=k, n=len(X),
            t_init=t_init, t_lloyd=t_lloyd, t_total=t_init+t_lloyd,
            mem_init_kb=mem_init/1024, mem_lloyd_kb=mem_lloyd/1024,
            inertia=inertia_hist[-1] if inertia_hist else None,
            inertia_history=inertia_hist,
            n_iter=len(inertia_hist),
            labels=labels
        )
        if best is None or result['inertia'] < best['inertia']:
            best = result

    return best


# ══════════════════════════════════════════════════════════════
#  EXPERIMENT A — Large-scale quality + convergence
#  n=200,000 samples, k in [10, 25, 50, 75, 100]
# ══════════════════════════════════════════════════════════════
print("=" * 65)
print("EXP A: Large-scale  n=200,000  k=[10,25,50,75,100]")
print("=" * 65)

N_LARGE  = 200_000
K_VALUES = [10, 25, 50, 75, 100]

X_large, _ = make_blobs(n_samples=N_LARGE, centers=100, cluster_std=1.5,
                        random_state=42, n_features=20)
X_large = StandardScaler().fit_transform(X_large)

# Init functions (wrapped to accept (X, k))
inits = {
    "OriginScale":    lambda X, k: X[np.argsort(np.linalg.norm(X, axis=1))[:k]].copy(),
    "KMeans++":       lambda X, k: KMeans(k, init='k-means++', n_init=1, max_iter=1,
                                          random_state=42).fit(X).cluster_centers_,
    "Random":         lambda X, k: X[np.random.RandomState(42).choice(len(X), k, replace=False)].copy(),
    "MiniBatch init": lambda X, k: MiniBatchKMeans(k, n_init=1, random_state=42,
                                                    max_iter=1).fit(X).cluster_centers_,
    "PCA-based":      pca_init,
}

expA_rows  = []
conv_data  = {}   # name -> {k -> inertia_history}

for k in K_VALUES:
    print(f"\n  k={k}")
    for name, init_fn in inits.items():
        res = run_full(X_large, k, init_fn, name)
        expA_rows.append({
            "k": k, "n": N_LARGE, "init": name,
            "t_init_ms":  round(res["t_init"]*1000, 2),
            "t_total_ms": round(res["t_total"]*1000, 2),
            "inertia":    round(res["inertia"], 1),
            "n_iter":     res["n_iter"],
            "mem_init_kb":round(res["mem_init_kb"], 1),
        })
        if name not in conv_data: conv_data[name] = {}
        conv_data[name][k] = res["inertia_history"]
        print(f"    {name:18s}  t_init={res['t_init']*1000:7.2f}ms  "
              f"iters={res['n_iter']:3d}  inertia={res['inertia']:.0f}")

df_A = pd.DataFrame(expA_rows)

# ══════════════════════════════════════════════════════════════
#  EXPERIMENT B — Init time vs k (pure init, no Lloyd)
#  n=200,000, k from 3 to 200
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("EXP B: Init time vs k  (pure initialization, n=200,000)")
print("=" * 65)

K_SWEEP    = [3, 5, 10, 25, 50, 75, 100, 150, 200]
INIT_REPS  = 5
expB_rows  = []

for k in K_SWEEP:
    for name, init_fn in inits.items():
        times = []
        for _ in range(INIT_REPS):
            t0 = time.perf_counter()
            init_fn(X_large, k)
            times.append(time.perf_counter() - t0)
        t_min = min(times)
        expB_rows.append({"k": k, "init": name, "t_ms": round(t_min*1000, 3)})
        print(f"  k={k:3d}  {name:18s}  {t_min*1000:.3f}ms")

df_B = pd.DataFrame(expB_rows)

# ══════════════════════════════════════════════════════════════
#  EXPERIMENT C — Inertia comparison at k=50, n=200k (focus)
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("EXP C: Inertia + convergence detail  k=50  n=200,000")
print("=" * 65)

K_FOCUS = 50
expC_rows = []
conv_k50  = {}

for name, init_fn in inits.items():
    res = run_full(X_large, K_FOCUS, init_fn, name, reps=1)
    expC_rows.append({
        "init": name, "k": K_FOCUS,
        "final_inertia":   round(res["inertia"], 1),
        "n_iter":          res["n_iter"],
        "t_init_ms":       round(res["t_init"]*1000, 2),
        "t_total_ms":      round(res["t_total"]*1000, 2),
        "mem_init_kb":     round(res["mem_init_kb"], 1),
    })
    conv_k50[name] = res["inertia_history"]
    print(f"  {name:18s}  inertia={res['inertia']:.0f}  iters={res['n_iter']}  "
          f"t_init={res['t_init']*1000:.1f}ms")

df_C = pd.DataFrame(expC_rows)

# ══════════════════════════════════════════════════════════════
#  COMPLEXITY TABLE  (theoretical, for paper section)
# ══════════════════════════════════════════════════════════════
complexity = [
    {"Initializer": "OriginScale",    "Time":  "O(n log n)",        "Space": "O(n)",    "Deterministic": "Yes", "Param-free": "Yes"},
    {"Initializer": "KMeans++",       "Time":  "O(n·k)",            "Space": "O(n)",    "Deterministic": "No",  "Param-free": "No (seed)"},
    {"Initializer": "Random",         "Time":  "O(n)",              "Space": "O(k)",    "Deterministic": "No",  "Param-free": "No (seed)"},
    {"Initializer": "MiniBatch init", "Time":  "O(b·k·t)",          "Space": "O(b)",    "Deterministic": "No",  "Param-free": "No (seed, batch)"},
    {"Initializer": "PCA-based",      "Time":  "O(n·d·log k)",      "Space": "O(n·c)",  "Deterministic": "Yes", "Param-free": "Yes"},
]

# ══════════════════════════════════════════════════════════════
#  MEMORY PROFILING at n=200k, k=50
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("Memory profiling  n=200,000  k=50")
print("=" * 65)

mem_rows = []
for name, init_fn in inits.items():
    tracemalloc.start()
    init_fn(X_large, K_FOCUS)
    _, peak_kb = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    mem_rows.append({"init": name, "peak_kb": round(peak_kb/1024, 1)})
    print(f"  {name:18s}  peak={peak_kb/1024:.1f} KB")

df_mem = pd.DataFrame(mem_rows)

# ══════════════════════════════════════════════════════════════
#  PLOTS
# ══════════════════════════════════════════════════════════════
COLORS = {
    "OriginScale":    "#E63946",
    "KMeans++":       "#457B9D",
    "Random":         "#A8DADC",
    "MiniBatch init": "#F4A261",
    "PCA-based":      "#2A9D8F",
}
MARKERS = {"OriginScale":"o","KMeans++":"s","Random":"^","MiniBatch init":"D","PCA-based":"P"}
LW = 2.2

# ── Plot 1: Init time vs k ────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
for name in inits:
    sub = df_B[df_B["init"] == name].sort_values("k")
    ax.plot(sub["k"], sub["t_ms"], label=name,
            color=COLORS[name], marker=MARKERS[name],
            linewidth=LW, markersize=7)
ax.set_xlabel("Number of clusters  k", fontsize=13)
ax.set_ylabel("Initialization time (ms)", fontsize=13)
ax.set_title(f"Initialization Time vs k  (n = {N_LARGE:,})", fontsize=14, fontweight='bold')
ax.legend(fontsize=10, framealpha=0.9)
ax.set_yscale("log")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/plot1_init_time_vs_k.png", dpi=180, bbox_inches='tight')
plt.close()
print("\nSaved: plot1_init_time_vs_k.png")

# ── Plot 2: Final inertia at k=50 (bar chart) ────────────────
fig, ax = plt.subplots(figsize=(8, 5))
names   = df_C["init"].tolist()
inertias = df_C["final_inertia"].tolist()
colors  = [COLORS[n] for n in names]
bars    = ax.bar(names, inertias, color=colors, edgecolor='white', linewidth=1.2)
ax.set_ylabel("Final Inertia (within-cluster sum of squares)", fontsize=12)
ax.set_title(f"Final Inertia After Convergence  (n={N_LARGE:,}, k={K_FOCUS})", fontsize=13, fontweight='bold')
ax.set_xticklabels(names, rotation=15, ha='right', fontsize=11)
for bar, val in zip(bars, inertias):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:,.0f}", ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/plot2_final_inertia_k50.png", dpi=180, bbox_inches='tight')
plt.close()
print("Saved: plot2_final_inertia_k50.png")

# ── Plot 3: Convergence curves at k=50 ───────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
for name, hist in conv_k50.items():
    ax.plot(range(1, len(hist)+1), hist,
            label=f"{name} ({len(hist)} iters)",
            color=COLORS[name], linewidth=LW,
            marker=MARKERS[name], markevery=max(1, len(hist)//8), markersize=6)
ax.set_xlabel("Iteration", fontsize=13)
ax.set_ylabel("Inertia", fontsize=13)
ax.set_title(f"Convergence Curves  (n={N_LARGE:,}, k={K_FOCUS})", fontsize=13, fontweight='bold')
ax.legend(fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/plot3_convergence_k50.png", dpi=180, bbox_inches='tight')
plt.close()
print("Saved: plot3_convergence_k50.png")

# ── Plot 4: 2×2 summary panel ────────────────────────────────
fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, hspace=0.38, wspace=0.32)

# 4a — init time vs k (log scale)
ax0 = fig.add_subplot(gs[0, 0])
for name in inits:
    sub = df_B[df_B["init"] == name].sort_values("k")
    ax0.plot(sub["k"], sub["t_ms"], label=name,
             color=COLORS[name], marker=MARKERS[name], linewidth=LW, markersize=6)
ax0.set_yscale("log")
ax0.set_xlabel("k", fontsize=11); ax0.set_ylabel("Init time (ms, log)", fontsize=11)
ax0.set_title("(a) Init Time vs k", fontsize=12, fontweight='bold')
ax0.legend(fontsize=8); ax0.grid(True, alpha=0.3)

# 4b — iterations to convergence vs k
ax1 = fig.add_subplot(gs[0, 1])
for name in inits:
    sub = df_A[df_A["init"] == name].sort_values("k")
    ax1.plot(sub["k"], sub["n_iter"], label=name,
             color=COLORS[name], marker=MARKERS[name], linewidth=LW, markersize=6)
ax1.set_xlabel("k", fontsize=11); ax1.set_ylabel("Iterations to converge", fontsize=11)
ax1.set_title("(b) Iterations to Convergence vs k", fontsize=12, fontweight='bold')
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

# 4c — final inertia vs k
ax2 = fig.add_subplot(gs[1, 0])
for name in inits:
    sub = df_A[df_A["init"] == name].sort_values("k")
    ax2.plot(sub["k"], sub["inertia"], label=name,
             color=COLORS[name], marker=MARKERS[name], linewidth=LW, markersize=6)
ax2.set_xlabel("k", fontsize=11); ax2.set_ylabel("Final Inertia", fontsize=11)
ax2.set_title("(c) Final Inertia vs k", fontsize=12, fontweight='bold')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

# 4d — memory usage at k=50
ax3 = fig.add_subplot(gs[1, 1])
mnames  = df_mem["init"].tolist()
mvals   = df_mem["peak_kb"].tolist()
mcolors = [COLORS[n] for n in mnames]
bars3   = ax3.bar(mnames, mvals, color=mcolors, edgecolor='white', linewidth=1.2)
ax3.set_ylabel("Peak memory (KB)", fontsize=11)
ax3.set_title(f"(d) Peak Memory Usage  (k={K_FOCUS})", fontsize=12, fontweight='bold')
ax3.set_xticklabels(mnames, rotation=20, ha='right', fontsize=9)
ax3.grid(axis='y', alpha=0.3)

fig.suptitle(f"OriginScale Large-Scale Analysis  —  n={N_LARGE:,} samples, d=20 features",
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig(f"{FIGDIR}/plot4_summary_panel.png", dpi=180, bbox_inches='tight')
plt.close()
print("Saved: plot4_summary_panel.png")

# ── Plot 5: Speedup heatmap (init time ratio vs KMeans++) ────
fig, ax = plt.subplots(figsize=(8, 4))
pivot = df_B.pivot(index="init", columns="k", values="t_ms")
pp_row = pivot.loc["KMeans++"]
ratio  = pivot.div(pivot.loc["OriginScale"])   # ratio = other / OriginScale
ratio  = ratio.drop(index="OriginScale")        # remove self-row
sns.heatmap(ratio, annot=True, fmt=".1f", cmap="RdYlGn",
            linewidths=0.5, ax=ax, cbar_kws={"label": "Time ratio vs OriginScale (higher = slower)"})
ax.set_title("Init Time Ratio vs OriginScale  (values > 1 mean OriginScale is faster)",
             fontsize=11, fontweight='bold')
ax.set_xlabel("k"); ax.set_ylabel("")
plt.tight_layout()
plt.savefig(f"{FIGDIR}/plot5_speedup_heatmap.png", dpi=180, bbox_inches='tight')
plt.close()
print("Saved: plot5_speedup_heatmap.png")

# ══════════════════════════════════════════════════════════════
#  SAVE RESULTS AS JSON
# ══════════════════════════════════════════════════════════════
# strip inertia_history from json (too large) — keep only summary
results = {
    "expA_large_scale":  expA_rows,
    "expB_init_vs_k":    expB_rows,
    "expC_k50_detail":   expC_rows,
    "complexity_table":  complexity,
    "memory_profile":    mem_rows,
    "conv_k50_lengths":  {name: len(hist) for name, hist in conv_k50.items()},
    "conv_k50_first5":   {name: hist[:5]  for name, hist in conv_k50.items()},
    "conv_k50_last3":    {name: hist[-3:] for name, hist in conv_k50.items()},
}

df_A.to_csv("/kaggle/working/expA_large_scale.csv",   index=False)
df_B.to_csv("/kaggle/working/expB_init_vs_k.csv",     index=False)
df_C.to_csv("/kaggle/working/expC_k50_detail.csv",    index=False)
df_mem.to_csv("/kaggle/working/memory_profile.csv",   index=False)
pd.DataFrame(complexity).to_csv("/kaggle/working/complexity.csv", index=False)

print("\n\n" + "=" * 65)
print("PASTE THIS JSON BACK TO CLAUDE:")
print("=" * 65)
print(json.dumps(results, indent=2))
print("\n\nPlots saved:")
for f in ["plot1_init_time_vs_k.png","plot2_final_inertia_k50.png",
          "plot3_convergence_k50.png","plot4_summary_panel.png",
          "plot5_speedup_heatmap.png"]:
    print(f"  /kaggle/working/{f}")

EXP A: Large-scale  n=200,000  k=[10,25,50,75,100]

  k=10
    OriginScale         t_init=  29.88ms  iters= 59  inertia=2855526
    KMeans++            t_init= 829.33ms  iters= 41  inertia=2836958
    Random              t_init=   9.21ms  iters= 62  inertia=2813582
    MiniBatch init      t_init= 163.02ms  iters= 18  inertia=2834763
    PCA-based           t_init=  93.46ms  iters= 74  inertia=2778721

  k=25
    OriginScale         t_init=  27.10ms  iters= 19  inertia=2162164
    KMeans++            t_init=1082.18ms  iters= 18  inertia=2102144
    Random              t_init=  10.00ms  iters= 17  inertia=2136911
    MiniBatch init      t_init= 222.69ms  iters= 42  inertia=2072003
    PCA-based           t_init= 134.88ms  iters= 16  inertia=2063937

  k=50
    OriginScale         t_init=  26.33ms  iters= 39  inertia=1487112
    KMeans++            t_init=1419.00ms  iters=  9  inertia=1291883
    Random              t_init=  10.10ms  iters=  8  inertia=1356004
    MiniBatch init      t_in